# PulmoScan: inspect saved v2 logits — no training

This notebook inspects the **already selected DenseNet121 checkpoint**. It never fits a model,
chooses another model, searches thresholds, or changes original artifacts. It does not download pretrained weights.

Attach **two inputs**:
1. The original **paultimothymooney/chest-xray-pneumonia** dataset.
2. Your saved v2 notebook output containing suite **kaggle-v2-20261110T094413Z**. Alternatively,
   upload **pulmoscan_models (3).zip** as a private Kaggle dataset and attach it. ZIP and unpacked inputs are supported.

Select an available GPU for inference. Internet is not needed if Kaggle's usual libraries are present.
Import this notebook into a **new notebook/session**, not into the original training notebook.
Use **Run All** here: there is no training cell. Keep your original training outputs unchanged.

The existing benchmark is already reviewed. These diagnostics cannot establish new unbiased final accuracy.
Download **pulmoscan_diagnostics.zip** when complete and save the notebook version including outputs.

In [ ]:
import base64, hashlib, importlib.util, json, sys, zlib
from pathlib import Path, PurePosixPath
WORKSPACE = Path('/kaggle/working/pulmoscan_diagnostics')
SOURCE_INDEX = {'src/compute.py': '14e5d99ba467b12c2352d19c0272498d53a0ad537119211208422e3b9bd8ea6a', 'src/data_protocol.py': '2df0eb1dd2d353cf2a0cb61edd8210e4740bd6627a3fb51ed7a4c0693ad3b88a', 'src/modeling.py': 'f19897e3415d385c569275b0da3b663c8caf70fa16d7b9e29348010463003ef9', 'src/scoring.py': '8b221e04257f9a3200b56c2e92ce11c574e8c3818193fe54911d3a1e74a27d19', 'scripts/experiment_bundle.py': '2b5f0b86a54483df6ed53c957f8a963ee2859d8053ffee7700e18001a2311106', 'scripts/inspect_checkpoint.py': '65b6c6e283d52c7b7cf04db3a698c3861890e3c44d130a38fcf604ba97605931', 'kaggle/diagnostic_runtime.py': '9199cc008e63d46daa795cdcec637ec1fc39c28d87a83b202dc85f04beb199b4', 'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'scripts/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'}
SOURCE_PAYLOAD = '...'
packed = base64.b64decode(SOURCE_PAYLOAD)
if hashlib.sha256(packed).hexdigest() != '777368564bd616db538ebedc41624d93e977a76d9cd0b80d4aa9774708587765':
    raise ValueError('Embedded diagnostic source checksum failed.')
files = json.loads(zlib.decompress(packed))
if set(files) != set(SOURCE_INDEX):
    raise ValueError('Embedded source index differs.')
for name, encoded in files.items():
    relative = PurePosixPath(name)
    if relative.is_absolute() or '..' in relative.parts or '\\' in name or ':' in name:
        raise ValueError('Unsafe embedded path.')
    content = base64.b64decode(encoded)
    if hashlib.sha256(content).hexdigest() != SOURCE_INDEX[name]:
        raise ValueError('Embedded source checksum differs: ' + name)
    target = WORKSPACE.joinpath(*relative.parts)
    if not target.resolve().is_relative_to(WORKSPACE.resolve()):
        raise ValueError('Source escaped the diagnostic workspace.')
    if target.exists() and target.read_bytes() != content:
        raise ValueError('Existing diagnostic workspace uses different code. Choose a new WORKSPACE.')
for name, encoded in files.items():
    target = WORKSPACE / name
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.write_bytes(base64.b64decode(encoded))
index_path = WORKSPACE / 'source_index.json'
index_path.write_text(json.dumps(SOURCE_INDEX, sort_keys=True, indent=2))
sys.path.insert(0, str(WORKSPACE))
print('Verified inference-only source files:', len(files))

## Settings and input detection

The default suite is the reviewed v2 export. Auto-detection accepts duplicate representations of the same
checkpoint, but refuses distinct matching models. Set paths explicitly if needed.
`INCLUDE_REVIEWED_BENCHMARK=True` is deliberate: it investigates the already observed errors.
Set False for validation/calibration only. The original saved threshold remains unchanged.
`PRECISION='saved'` follows the checkpoint's AMP preference on CUDA; device/library changes may affect reproduction.

In [ ]:
from datetime import datetime, timezone
SUITE_NAME = 'kaggle-v2-20261110T094413Z'
ARTIFACT_PATH = None  # ZIP, unpacked bundle root, or saved suite folder containing comparison.json.
DATA_ROOT = None  # Directory directly containing train/, val/, test/.
REQUIRE_GPU = True  # Set False deliberately for automatic GPU/CPU selection.
INCLUDE_REVIEWED_BENCHMARK = True
PRECISION = 'saved'
BATCH_SIZE = 16
DIAGNOSTIC_NAME = 'inspection-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')

for module in ('torch', 'torchvision', 'numpy', 'sklearn', 'PIL'):
    if importlib.util.find_spec(module) is None:
        raise RuntimeError('Missing runtime library: ' + module + '. Use a complete Kaggle Python GPU runtime.')
import torch, torchvision
from src.compute import select_device
device = select_device('cuda' if REQUIRE_GPU else 'auto')
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| device', device)
spec = importlib.util.spec_from_file_location('pulmoscan_diagnostic_runtime', WORKSPACE / 'kaggle/diagnostic_runtime.py')
runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime)
artifact = runtime.discover_artifact('/kaggle/input', SUITE_NAME, ARTIFACT_PATH)
data_root = runtime.discover_dataset('/kaggle/input', DATA_ROOT)
print('Source dataset:', data_root)

## Fixed-checkpoint inference

Verify the checkpoint checksum, original manifest, split boundaries, and all original source image hashes.
A different mount path is allowed only when the original bytes still match; the manifest is not rewritten.
Then record both logits, their difference, historical float32 softmax scores, stable log probabilities,
and per-image log loss. No optimizer or gradient updates occur.

AUC from the logit difference can retain ordering lost when probabilities round to exactly 1.0.
This does **not** repair wrong predictions at the saved threshold or calibrate model probabilities.
Reference comparisons report any changed decisions instead of claiming exact reproduction across runtimes.

In [ ]:
from scripts.inspect_checkpoint import inspect, export_diagnostics
output = WORKSPACE / 'results' / DIAGNOSTIC_NAME
report = inspect(artifact, data_root, output,
                 include_reviewed_benchmark=INCLUDE_REVIEWED_BENCHMARK,
                 batch_size=BATCH_SIZE,
                 device_preference='cuda' if REQUIRE_GPU else 'auto',
                 precision=PRECISION, plots=True)
for split, result in report['splits'].items():
    original = result['metrics_at_saved_threshold']
    print(f"{split}: probability AUC={original['roc_auc']:.6f} | margin AUC={result['roc_auc_logit_margin']:.6f} | "
          f"accuracy at SAVED cut={original['accuracy']:.2%} | loss={result['loss_from_logits']:.6f}")
    print('Saturation:', json.dumps(result['score_saturation']))
    print('Reference agreement:', json.dumps(result.get('reference_comparison', {})))

## Inspect images and download

The gallery shows normal images incorrectly flagged at the unchanged threshold, ordered by raw margin.
These are model errors on public benchmark images, not clinical diagnoses. Raw-margin AUC and stable
log loss are diagnostics; do not use this reviewed benchmark to choose another model or threshold.

The ZIP contains the report, per-image CSVs, plots, source checksums, and a checksum index.
It does not contain another trained model. Download it and send it back for review.

In [ ]:
from IPython.display import Image as DisplayImage, FileLink, display
for filename in ('logit_distributions.png', 'false_positive_examples.png'):
    if (output / filename).exists():
        display(DisplayImage(filename=str(output / filename), width=1100))
(output / 'diagnostic_source_index.json').write_text(json.dumps(SOURCE_INDEX, sort_keys=True, indent=2))
download = Path('/kaggle/working/pulmoscan_diagnostics.zip')
if download.exists():
    download = download.with_name('pulmoscan_diagnostics_' + DIAGNOSTIC_NAME + '.zip')
export_diagnostics(output, download)
print('Download:', download)
display(FileLink(str(download)))
print('Also available in Kaggle output files. Save Version including outputs before ending the session.')